# Scenario Simulation & Stress Testing

In this notebook, we test the robustness of the proposed inventory
replenishment policy under different real-world stress scenarios.

The goal is to evaluate:
- Service level impact
- Stockout risk
- Inventory exposure under uncertainty

In [6]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

import sys
sys.path.append('../')

DATA_PATH_PROCESSED = "../data/processed"

In [7]:
df = pd.read_csv(f"{DATA_PATH_PROCESSED}/feature_engineered_with_segments.csv")
print("Segmented data loaded successfully")

inventory_policy = pd.read_csv(f"{DATA_PATH_PROCESSED}/inventory_replenishment_policy.csv")
print("Inventory replenishment policy data loaded successfully")

Segmented data loaded successfully
Inventory replenishment policy data loaded successfully


## Baseline Inventory Position

We merge the current inventory levels with the optimized
replenishment policy to evaluate stock health.

In [8]:
current_inventory = (
    df.groupby("sku_id")["units_sold"]
    .mean()
    .reset_index(name="avg_weekly_demand")
)

baseline = inventory_policy.merge(
    current_inventory,
    on="sku_id",
    how="left"
)

baseline.head()

,sku_id,avg_weekly_demand_x,std_weekly_demand,avg_lead_time,std_lead_time,safety_stock,reorder_point,SKU_segment,segment_multiplier,adjusted_safety_stock,adjusted_reorder_point,avg_weekly_demand_y
0,SKU0001,30.737569,6.840500,10.0,0.0,37.870069,345.245760,AX,1.2,45.444083,352.819774,30.737569
1,SKU0002,31.842541,8.140393,6.0,0.0,34.908346,225.963594,BX,1.0,34.908346,225.963594,31.842541
2,SKU0003,25.698895,7.416792,5.0,0.0,29.034169,157.528644,CX,0.8,23.227335,151.721810,25.698895
3,SKU0004,28.801105,7.211425,5.0,0.0,28.230228,172.235752,BX,1.0,28.230228,172.235752,28.801105
4,SKU0005,34.751381,7.132234,5.0,0.0,27.920223,201.677129,AX,1.2,33.504267,207.261173,34.751381


## Simulation Framework

We simulate weekly demand and compare it against:
- Current inventory
- Reorder point thresholds

A stockout occurs when simulated demand exceeds available inventory.

In [9]:
def simulate_inventory(demand_mean, demand_std, inventory_level, weeks=8):
    stockouts = 0
    inventory = inventory_level

    for _ in range(weeks):
        demand = max(0, np.random.normal(demand_mean, demand_std))
        inventory -= demand

        if inventory < 0:
            stockouts += 1
            inventory = 0

    return stockouts

## Scenario 1: Normal Demand Conditions

Baseline scenario using historical demand statistics.

In [10]:
baseline["normal_stockouts"] = baseline.apply(
    lambda row: simulate_inventory(
        row["avg_weekly_demand_x"],
        row["std_weekly_demand"],
        row["adjusted_reorder_point"]
    ),
    axis=1
)

## Scenario 2: Demand Surge (+30%)

Simulates festive season or aggressive promotional campaigns.

In [12]:
baseline["surge_stockouts"] = baseline.apply(
    lambda row: simulate_inventory(
        row["avg_weekly_demand_x"] * 1.3,
        row["std_weekly_demand"],
        row["adjusted_reorder_point"]
    ),
    axis=1
)

## Scenario 3: Supplier Delay (+50% Lead Time)

Simulates unexpected supplier delays.

In [13]:
baseline["delay_stockouts"] = baseline.apply(
    lambda row: simulate_inventory(
        row["avg_weekly_demand_x"],
        row["std_weekly_demand"],
        row["adjusted_reorder_point"] * 0.7  # effective availability reduced
    ),
    axis=1
)

## Scenario Comparison Summary

In [14]:
scenario_summary = baseline[
    ["SKU_segment", "normal_stockouts", "surge_stockouts", "delay_stockouts"]
]

scenario_summary.groupby("SKU_segment").mean()

,normal_stockouts,surge_stockouts,delay_stockouts
SKU_segment,,,
AX,1.739130,2.478261,2.782609
BX,1.300000,2.600000,3.000000
CX,1.571429,2.285714,2.714286


## Service Level Approximation

Service level is approximated as:

1 - (stockout_weeks / total_weeks)

In [15]:
weeks_simulated = 8

baseline["service_level_normal"] = (
    1 - baseline["normal_stockouts"] / weeks_simulated
)

baseline["service_level_surge"] = (
    1 - baseline["surge_stockouts"] / weeks_simulated
)

baseline["service_level_delay"] = (
    1 - baseline["delay_stockouts"] / weeks_simulated
)

baseline[
    [
        "SKU_segment",
        "service_level_normal",
        "service_level_surge",
        "service_level_delay"
    ]
].groupby("SKU_segment").mean()

,service_level_normal,service_level_surge,service_level_delay
SKU_segment,,,
AX,0.782609,0.690217,0.652174
BX,0.837500,0.675000,0.625000
CX,0.803571,0.714286,0.660714


## Final Conclusion

This project demonstrates how demand forecasting can be operationalized
into a practical inventory replenishment system.

Key outcomes:
- Reduced stockout risk for high-value SKUs
- Lower excess inventory for long-tail products
- Clear, explainable replenishment rules
- Scenario-tested robustness

The final outputs are directly usable by supply chain and
warehouse planning teams.